# v7 Python Authoring Artifact Walkthrough

This notebook is for inspecting a run created by the Python authoring layer.

Use it after running:
- `jupyter lab notebooks/v7_python_authoring_quickstart.ipynb`, or
- `python3 version/v7/examples/python_authoring_tiny_lm_v7.py`

The goal is to make the handoff boundary explicit: Python authored the plan, but `v7` created the manifest, IR, codegen outputs, and generated C runtime.


In [ ]:
from pathlib import Path
import json

RUN_DIR = Path.home() / ".cache" / "ck-engine-v7" / "models" / "train" / "python-ui-notebook-demo"
RUN_DIR


In [ ]:
plan = json.loads((RUN_DIR / "python_authoring_plan.json").read_text(encoding="utf-8"))
plan.keys()


In [ ]:
{
    "template": plan["template"],
    "model": plan["model"],
    "tokenizer": plan["tokenizer"],
    "artifacts": plan["artifacts"],
}


In [ ]:
manifest = json.loads((RUN_DIR / "weights_manifest.json").read_text(encoding="utf-8"))
manifest.keys()


In [ ]:
{
    "model": manifest.get("config", {}).get("model"),
    "template_name": manifest.get("template", {}).get("name"),
    "num_entries": len(manifest.get("entries", [])),
    "artifacts": manifest.get("artifacts", {}),
}


In [ ]:
ir1 = json.loads((RUN_DIR / "ir1_train_forward.json").read_text(encoding="utf-8"))
ir2 = json.loads((RUN_DIR / "ir2_train_backward.json").read_text(encoding="utf-8"))
layout = json.loads((RUN_DIR / "layout_train.json").read_text(encoding="utf-8"))

{
    "ir1_keys": list(ir1.keys())[:10],
    "ir2_keys": list(ir2.keys())[:10],
    "layout_keys": list(layout.keys())[:10],
}


In [ ]:
train_report_path = RUN_DIR / "train_e2e_latest.json"
if train_report_path.exists():
    train_report = json.loads(train_report_path.read_text(encoding="utf-8"))
    summary = {
        "top_level_keys": list(train_report.keys())[:20],
    }
else:
    summary = {"note": "train_e2e_latest.json not found yet"}

summary


## Reading The Pipeline

The current flow is:
1. Python writes the project plan and optional embedded template file.
2. `ck_run_v7.py init` creates `weights.bump` and `weights_manifest.json`.
3. `v7` scripts lower the manifest + template into `ir1`, `ir2`, and `layout`.
4. `codegen_train_runtime_v7.py` emits `generated_train_runtime_v7.c`.
5. The generated runtime is compiled to `libtrain.so`.
6. `ck_run_v7.py train` executes training and writes telemetry and JSON reports.
7. Python can optionally call the existing viewer tools to refresh `ir_report.html`, export viewer-side artifacts, and regenerate `ir_hub.html`.

For tiny inline-text runs, `ir_report.html`, `embeddings.json`, and `ir_hub.html` are the usual outputs. `dataset_viewer.html` and `attention.json` stay conditional on dataset/tokenizer artifacts.
So the notebook-friendly Python layer is real, but the production execution surface is still generated C.
